# Demo 06A · Notebook 1: Participant Hybrid Retrieval

You run one predictable, graph-enriched hotel retrieval path against **your own Neo4j Aura instance** and **Amazon Bedrock**. This notebook creates no AWS resources.

## Who owns what

| Neo4j owns | AWS owns |
|---|---|
| Connected hotel knowledge: hotels, amenities, ratings, policies | Amazon Bedrock reasons over the retrieved evidence |
| The vector index `hotel_chunk_embeddings` and full-text index `hotel_chunk_fulltext` | Amazon Nova 2 creates the query embedding |
| The reviewed Cypher traversal that enriches a matched chunk with its hotel |  |

The retriever is one fixed `HybridCypherRetriever`: explicit `NAIVE` fusion, `top_k=5`, one reviewed traversal. It accepts only a `query`. There is no ranker, alpha, or retriever-mode selector here; those comparisons live in Demo 01b.

In [ ]:
import json
import os

import boto3
from dotenv import load_dotenv

from contracts import MAX_GUESTS, OVER_LIMIT_GUESTS
from graph_setup import (
    HERO_NAME,
    apply_demo6_graph,
    load_manifest,
    readiness_problems,
)
from hybrid_retrieval import (
    GROUNDING_INSTRUCTIONS,
    Neo4jConfig,
    search_hotel_knowledge,
)
from neo4j import GraphDatabase

load_dotenv()

NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_READY = all(os.getenv(name) for name in NEO4J_ENV)
BEDROCK_READY = boto3.Session().get_credentials() is not None
RETRIEVAL_READY = NEO4J_READY and BEDROCK_READY

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_ID = os.getenv("MODEL_ID", "us.anthropic.claude-sonnet-4-6")
HERO_QUESTION = f"What amenities and guest rating does {HERO_NAME} have?"
AVAILABILITY_QUESTION = f"Does {HERO_NAME} guarantee room availability next weekend?"

if not NEO4J_READY:
    print("Neo4j is not configured (set NEO4J_URI/USERNAME/PASSWORD/DATABASE); live cells will be skipped.")
if not BEDROCK_READY:
    print("AWS credentials are not configured; live cells will be skipped.")
if RETRIEVAL_READY:
    print("Participant retrieval is configured. Ready to run the hero question.")

## 1. Confirm your Aura connection and prepared indexes

The cell below prepares only the Demo 06 graph-owned data (fixture hotel IDs, uniqueness constraints, and the maximum-guests rule) idempotently, then verifies that both retrieval indexes are online and the hero hotel is present. It never modifies canonical hotel facts.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph preparation: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        driver.verify_connectivity()
        apply_demo6_graph(driver, config.database, manifest)
        problems = readiness_problems(driver, config.database, manifest)
    finally:
        driver.close()
    if problems:
        print("Graph is not ready:")
        for problem in problems:
            print(f"  - {problem}")
    else:
        print("Participant graph is ready: both indexes online, fixtures applied, rule present.")

## 2. The hero question

> **What amenities and guest rating does AnyCompany Cairo Nile View have?**

The exact hotel name benefits from the full-text arm, the request for amenities and a rating benefits from semantic vector matching, and the reviewed traversal adds the connected hotel, its amenities, its rating, and the stable `hotel_id`. This is top-k grounded retrieval, not an exhaustive database listing.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping hero question: retrieval is not configured.")
else:
    results = search_hotel_knowledge(HERO_QUESTION)
    top = results[0]
    print(f"Question: {HERO_QUESTION}\n")
    print(f"Hotel: {top['hotel_name']}  (hotel_id={top['hotel_id']})")
    print(f"Combined hybrid score: {top['combined_score']:.4f}")
    print(f"Guest rating: {top['guest_rating']}")
    print(f"Exact matched terms: {', '.join(top['exact_terms']) or 'none'}")
    print(f"Amenities ({len(top['amenities'])}): {', '.join(top['amenities'])}")
    print("\nChunk evidence:")
    print(top["chunk_evidence"][:600])

### What just happened

- **Vector matching** handled the paraphrased request for "amenities and guest rating".
- **Full-text matching** locked onto the exact hotel name and location terms.
- The **reviewed Cypher traversal** followed the matched chunk to its hotel and returned up to 12 connected amenities, the guest rating, and the opaque `hotel_id`. That `hotel_id`, never the display name, is the identity the reservation command accepts in Notebook 2.

We do not tune the ranker, alpha, or weights here. The fusion behavior and `top_k` are fixed so the result is deterministic for the workshop.

## 3. A question the evidence cannot answer

> **Does AnyCompany Cairo Nile View guarantee room availability next weekend?**

The graph holds hotel knowledge, not live inventory. A grounded agent must **abstain** rather than invent availability. Below we wrap the same one retrieval tool in a small local agent with the workshop's grounding instructions, ask the grounded hero question, then ask the unanswerable one.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping grounded agent: retrieval is not configured.")
else:
    from strands import Agent, tool
    from strands.models import BedrockModel

    @tool
    def search_hotel_knowledge_tool(query: str) -> str:
        """Search grounded hotel evidence and return bounded JSON facts."""
        return json.dumps(search_hotel_knowledge(query), ensure_ascii=False)

    grounded_agent = Agent(
        model=BedrockModel(
            model_id=MODEL_ID, region_name=AWS_REGION, temperature=0
        ),
        tools=[search_hotel_knowledge_tool],
        system_prompt=(
            "You are a grounded hotel-information assistant. Call "
            "search_hotel_knowledge_tool before answering any hotel question.\n\n"
            + GROUNDING_INSTRUCTIONS
        ),
    )
    print(grounded_agent(HERO_QUESTION))

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping availability question: retrieval is not configured.")
else:
    print(grounded_agent(AVAILABILITY_QUESTION))

## Where this goes next

- **Demo 01b** compares vector, hybrid, Vector-Cypher, and Text2Cypher retrieval patterns. This notebook deliberately fixes one pattern instead of comparing them.
- **Notebook 2 (`02_agentcore_walkthrough.ipynb`)** is the facilitator-only walkthrough: the same retriever contract runs in a pre-deployed AgentCore Runtime, paired with one protected reservation command that rejects a 15-guest request and idempotently records a valid one.

You used your own Aura instance for everything above. You do not deploy or invoke the shared Runtime; you observe the facilitator run it.